# Swerve source-data audit
Run all cells from this checkout. Only Python standard-library modules are needed.
This verifies historical recorded responses, not representativeness or current opinion. See `README.md` and the full ledger for interpretation.

In [1]:
from pathlib import Path
import importlib.util, json
root = Path.cwd()
while not (root / "scripts/import-surveys.py").exists():
    if root.parent == root: raise RuntimeError("Open inside the Swerve checkout")
    root = root.parent
spec = importlib.util.spec_from_file_location("surveys", root / "scripts/import-surveys.py")
surveys = importlib.util.module_from_spec(spec)
spec.loader.exec_module(surveys)
outputs, audit = surveys.build()
print(json.dumps(audit, indent=2))

{
  "version": "surveys-v1",
  "sourceCommit": "4c1ff5e3aef1816ae04af63218015066e186c147",
  "verifiedQuestions": 109,
  "scheduledDays": 100,
  "scheduledSlots": 500,
  "distinctDailySets": 100,
  "repeatedQuestionSlots": 391,
  "categories": {
    "Food": 42,
    "Flying": 18,
    "Screen": 26,
    "Everyday": 10,
    "Traditions": 13
  },
  "minValidAnswers": 213,
  "sourceCount": 6,
  "currentOrRepresentativeClaim": false,
  "limits": [
    "Historical response frequencies, not current public opinion.",
    "Unweighted item-specific valid response bases, not all respondents.",
    "Some questions repeat; 100 distinct daily sets do not mean 500 distinct questions.",
    "No multi-select blocks, demographic subdivisions, or simulated people are imported."
  ]
}


In [2]:
ledger = json.loads(outputs["docs/content/survey-ledger.json"])
for item in ledger:
    assert sum(option["count"] for option in item["options"]) == item["validAnswers"]
    assert item["validAnswers"] + item["missingAnswers"] == item["respondentCount"]
    assert all(option["pct"] == 100 * option["count"] / item["validAnswers"] for option in item["options"])
    assert item["validAnswers"] >= 200
for path, generated in outputs.items():
    assert (root / path).read_text() == generated, path
print(f"All {len(ledger)} items reconcile; pinned evidence and generated files match.")

All 109 items reconcile; pinned evidence and generated files match.


In [3]:
example = next(item for item in ledger if item["questionId"] == "comma-survey-01")
print(example["originalWording"])
print(example["source"]["url"])
for option in example["options"]:
    print(f"{option['label']}: {option['count']}/{example['validAnswers']} = {option['pct']:.6f}%")
print("No percentage presets, missing-as-No conversion, or option merging.")

In your opinion, which sentence is more gramatically correct?
https://github.com/fivethirtyeight/data/blob/4c1ff5e3aef1816ae04af63218015066e186c147/comma-survey/comma-survey.csv
Without Oxford comma: 488/1129 = 43.224092%
With Oxford comma: 641/1129 = 56.775908%
No percentage presets, missing-as-No conversion, or option merging.


## Interpretation limits
109 distinct questions fill 500 round slots across 100 distinct daily sets; reuse is intentional and disclosed.
These 2014–2015 data use item-specific, unweighted valid-answer bases. They are not today’s players or a current population estimate. Exact field dates are unknown except where documented (Thanksgiving: 2015-11-17).
No margins of error or national-population inference are calculated. The 200-answer floor is an editorial rule, not a guarantee of sampling quality.